In [1]:
print("AI Assignment, General purpose AI reasoning system")

AI Assignment, General purpose AI reasoning system


In [2]:
%pip -q install datasets huggingface_hub pandas tqdm

In [3]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Optional, Any
from collections import defaultdict
import re
import json
import time
import os
import pandas as pd

RANDOM_SEED = 42
RULETAKER_LIMIT = 500
FOLIO_LIMIT = None
COUNTERBENCH_FILES = [
    "data_balanced_alpha_V1.json",
    "data_balanced_backdoor_V2.json",
]

In [4]:
# downloading and inspecting 3 required punlic data set

from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json

# RuleTaker
ruletaker_train = load_dataset(
    "tasksource/ruletaker",
    split=f"train[:{RULETAKER_LIMIT}]"
)

try:
    ruletaker_dev = load_dataset("tasksource/ruletaker", split="dev")
    ruletaker_dev = ruletaker_dev.select(range(min(300, len(ruletaker_dev))))
except Exception as exc:
    print("Using training data for RuleTaker testing:", exc)
    ruletaker_dev = ruletaker_train.select(range(min(100, len(ruletaker_train))))

print("RuleTaker:", ruletaker_train.column_names)
print(ruletaker_train[0])


# FOLIO
folio_dataset = load_dataset("tasksource/folio")

print("\nFOLIO splits:", list(folio_dataset.keys()))

for name, data in folio_dataset.items():
    print(name, data.column_names, len(data))

folio_split_name = "train" if "train" in folio_dataset else list(folio_dataset.keys())[0]
folio_data = folio_dataset[folio_split_name]

if FOLIO_LIMIT is not None:
    folio_data = folio_data.select(range(min(FOLIO_LIMIT, len(folio_data))))

print(folio_data[0])


# CounterBench
counterbench_data = {}

for filename in COUNTERBENCH_FILES:
    path = hf_hub_download(
        repo_id="CounterBench/CounterBench",
        repo_type="dataset",
        filename=filename
    )

    with open(path, encoding="utf-8") as f:
        counterbench_data[filename] = json.load(f)

    data = counterbench_data[filename]
    print("\n", filename, type(data).__name__, len(data))
    if isinstance(data, list) and data:
        print(data[0])
    elif isinstance(data, dict) and data:
        key = next(iter(data))
        print(key, data[key])


README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

... (11 outputs left out)

data_balanced_alpha_V1.json:   0%|          | 0.00/557k [00:00<?, ?B/s]


 data_balanced_alpha_V1.json list 1000
{'question': 'Would Lumbo occur if not Ziklo instead of Ziklo?', 'given_info': 'We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.', 'answer': 'no', 'type': 'basic', 'question_id': '0', 'meta': {'story_id': 'nonsense_0', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 0, 'graph_id': 'graph5'}}


data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]


 data_balanced_backdoor_V2.json list 200
{'question': 'Would Lumbo occur if not Ziklo instead of Ziklo?', 'given_info': 'We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune', 'answer': 'yes', 'type': 'backdoor', 'question_id': '0', 'meta': {'story_id': 'nonsense_0', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 0, 'graph_id': 'graph5'}}


In [5]:
# working on logical representation

@dataclass(frozen=True)
class Literal:
    predicate: str
    args: tuple[str, ...]
    negated: bool = False

    def __str__(self):
        fact = f"{self.predicate}({', '.join(self.args)})"
        return f"NOT {fact}" if self.negated else fact


@dataclass(frozen=True)
class Rule:
    body: tuple[Literal, ...]
    head: Literal

    def __str__(self):
        return " AND ".join(map(str, self.body)) + " -> " + str(self.head)


def normalize_token(text):
    text = text.lower().strip()
    text = re.sub(r"^(the|a|an)\s+", "", text)
    text = re.sub(r"[^a-z0-9_ ]", "", text)
    return re.sub(r"\s+", "_", text)


def normalize_predicate(text):
    return normalize_token(text).title()


def is_variable(term):
    return term.startswith("?")


def substitute_literal(literal, substitution):
    args = tuple(substitution.get(arg, arg) for arg in literal.args)
    return Literal(literal.predicate, args, literal.negated)


In [6]:
# working on unification and rule matching


def unify(pattern, fact, substitution=None):
    if substitution is None:
        substitution = {}

    if pattern.predicate != fact.predicate or pattern.negated != fact.negated:
        return None

    if len(pattern.args) != len(fact.args):
        return None

    result = substitution.copy()

    for p, f in zip(pattern.args, fact.args):
        if is_variable(p):
            if p in result and result[p] != f:
                return None
            result[p] = f
        elif p != f:
            return None

    return result


def match_rule_body(body, facts, substitution=None):
    if substitution is None:
        substitution = {}

    if not body:
        return [substitution]

    matches = []

    for fact in facts:
        new_sub = unify(body[0], fact, substitution)

        if new_sub is not None:
            matches.extend(
                match_rule_body(body[1:], facts, new_sub)
            )

    return matches


In [7]:
# a simple natural language parser taken from the web


def split_sentences(text: str) -> list[str]:
    # Contexts in these benchmarks are generally sentence-delimited.
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", str(text).strip())
        if s.strip()
    ]


def parse_atom_clause(clause: str,
                      implicit_subject: Optional[str] = None
                      ) -> Optional[Literal]:
    clause = clause.strip().strip(" .")
    clause = re.sub(r"^(and|,)+\s*", "", clause, flags=re.I)
    clause = re.sub(r"\s+", " ", clause)

    # Pronoun / generic-subject unary predicates.
    m = re.fullmatch(
        r"(someone|somebody|something|a person|a thing|they|it)\s+(?:is|are)\s+(not\s+)?(.+)",
        clause, flags=re.I
    )
    if m:
        _, neg, predicate = m.groups()
        return Literal(normalize_predicate(predicate), ("?x",), bool(neg))

    # Fixed-entity unary predicate.
    m = re.fullmatch(r"(.+?)\s+is\s+(not\s+)?(.+)", clause, flags=re.I)
    if m:
        entity, neg, predicate = m.groups()
        entity = normalize_token(entity)
        predicate = normalize_predicate(predicate)
        if entity and predicate:
            return Literal(predicate, (entity,), bool(neg))

    # Binary relation: subject (does not) relation object.
    m = re.fullmatch(
        r"(.+?)\s+(does not|do not)?\s*(eats?|visits?|needs?|sees?|chases?|likes?|loves?|hates?|helps?|knows?|has|have|wants?|follows?|attacks?|supports?)\s+(.+)",
        clause, flags=re.I
    )
    if m:
        subject, neg_phrase, relation, obj = m.groups()
        if normalize_token(subject) in {"someone", "somebody", "something", "they", "it"}:
            subject = "?x"
        else:
            subject = normalize_token(subject)
        obj = normalize_token(obj)
        return Literal(
            normalize_predicate(relation),
            (subject, obj),
            bool(neg_phrase)
        )

    # Generic implicit unary condition, e.g. "not young".
    if implicit_subject is not None:
        m = re.fullmatch(r"(not\s+)?([a-zA-Z][a-zA-Z0-9 _-]*)", clause)
        if m:
            neg, predicate = m.groups()
            return Literal(normalize_predicate(predicate), (implicit_subject,), bool(neg))

    return None


def split_conditions(text: str) -> list[str]:
    # Simple conjunction splitting. Handles "P and not Q" and "P and Q".
    return [part.strip() for part in re.split(r"\s+and\s+", text, flags=re.I) if part.strip()]


def parse_rule_sentence(sentence: str) -> Optional[Rule]:
    s = sentence.strip().rstrip(". ")

    # Conditional form: If ... then ...
    m = re.fullmatch(r"If\s+(.+?)\s+then\s+(.+)", s, flags=re.I)
    if m:
        antecedent, consequent = m.groups()
        body_parts = split_conditions(antecedent)
        body = []

        for index, part in enumerate(body_parts):
            lit = parse_atom_clause(part, implicit_subject="?x")
            if lit is None:
                return None
            body.append(lit)

        head = parse_atom_clause(consequent, implicit_subject="?x")
        if head is None:
            return None
        return Rule(tuple(body), head)

    # Class rule: "Young, kind things are successful."
    m = re.fullmatch(
        r"([A-Za-z ,]+?)\s+(things|people|persons)\s+are\s+(not\s+)?(.+)",
        s, flags=re.I
    )
    if m:
        conditions_text, _, neg, consequent = m.groups()
        predicates = [p.strip() for p in conditions_text.split(",") if p.strip()]
        if not predicates:
            return None
        body = tuple(
            Literal(normalize_predicate(p), ("?x",), False)
            for p in predicates
        )
        head = Literal(normalize_predicate(consequent), ("?x",), bool(neg))
        return Rule(body, head)

    # "All young things are smart."
    m = re.fullmatch(r"All\s+(.+?)\s+(things|people|persons)\s+are\s+(not\s+)?(.+)", s, flags=re.I)
    if m:
        antecedent, _, neg, consequent = m.groups()
        body = (Literal(normalize_predicate(antecedent), ("?x",), False),)
        head = Literal(normalize_predicate(consequent), ("?x",), bool(neg))
        return Rule(body, head)

    return None


def parse_context(context: str):
    facts: set[Literal] = set()
    rules: list[Rule] = []
    unsupported: list[str] = []

    for sentence in split_sentences(context):
        rule = parse_rule_sentence(sentence)
        if rule is not None:
            rules.append(rule)
            continue

        fact = parse_atom_clause(sentence)
        if fact is not None:
            facts.add(fact)
        else:
            unsupported.append(sentence)

    return facts, rules, unsupported


def parse_query(question: str) -> Optional[Literal]:
    # RuleTaker questions are usually propositions such as "Alice is young."
    return parse_atom_clause(question)

In [8]:
# forward chaining and reasoning


def forward_chain(initial_facts, rules, max_rounds=100):
    facts = set(initial_facts)
    trace = []

    for round_no in range(1, max_rounds + 1):
        new_facts = set()

        for rule in rules:
            for sub in match_rule_body(rule.body, facts):
                conclusion = substitute_literal(rule.head, sub)

                if not any(is_variable(arg) for arg in conclusion.args):
                    if conclusion not in facts:
                        new_facts.add(conclusion)
                        trace.append({
                            "round": round_no,
                            "rule": str(rule),
                            "substitution": sub.copy(),
                            "derived": str(conclusion)
                        })

        if not new_facts:
            break

        facts.update(new_facts)

    return facts, trace


def predict_entailment(context, query):
    facts, rules, unsupported = parse_context(context)
    parsed_query = parse_query(query)

    if parsed_query is None:
        return {
            "prediction": None,
            "supported": False,
            "reason": "Query could not be parsed.",
            "unsupported_sentences": unsupported,
            "trace": [],
            "derived_facts": facts
        }

    closure, trace = forward_chain(facts, rules)
    supported = not unsupported

    return {
        "prediction": parsed_query in closure,
        "supported": supported,
        "reason": "Query checked against derived facts." if supported
                  else "Some context sentences could not be parsed.",
        "unsupported_sentences": unsupported,
        "trace": trace,
        "derived_facts": closure
    }


In [9]:
# testing a basic example

context = """
Bob is young.
Bob is kind.
If someone is young then they are smart.
If someone is smart and kind then they are successful.
"""

queries = [
    "Bob is young.",
    "Bob is smart.",
    "Bob is successful.",
    "Bob is rude."
]

for query in queries:
    result = predict_entailment(context, query)
    print(query, "->", result["prediction"])

assert predict_entailment(context, "Bob is young.")["prediction"] is True
assert predict_entailment(context, "Bob is smart.")["prediction"] is True
assert predict_entailment(context, "Bob is successful.")["prediction"] is True
assert predict_entailment(context, "Bob is rude.")["prediction"] is False

assert parse_atom_clause("they are smart", "?x") == Literal("Smart", ("?x",))

print("All tests passed.")


Bob is young. -> True
Bob is smart. -> True
Bob is successful. -> True
Bob is rude. -> False
All tests passed.


In [10]:
# Testing and evaluating RuleTaker


def rule_taker_gold(label):
    label = str(label).strip().lower()

    if label == "entailment":
        return True
    if label == "not entailment":
        return False

    raise ValueError("Unknown label: " + label)


def evaluate_ruletaker(dataset):
    records = []
    start = time.perf_counter()

    for i, example in enumerate(dataset):
        result = predict_entailment(example["context"], example["question"])

        records.append({
            "id": example.get("id", i),
            "gold": rule_taker_gold(example["label"]),
            "prediction": result["prediction"],
            "supported": result["supported"]
        })

    elapsed = time.perf_counter() - start
    df = pd.DataFrame(records)
    supported = df[df["supported"]]

    summary = {
        "examples": len(df),
        "parser_supported": len(supported),
        "parser_coverage": df["supported"].mean() if len(df) else 0,
        "accuracy_all": (df["gold"] == df["prediction"]).mean() if len(df) else None,
        "accuracy_supported_only": (
            (supported["gold"] == supported["prediction"]).mean()
            if len(supported) else None
        ),
        "total_inference_seconds": elapsed,
        "average_seconds_per_example": elapsed / len(df) if len(df) else 0
    }

    return summary, df


ruletaker_summary, ruletaker_results = evaluate_ruletaker(ruletaker_dev)

print(ruletaker_summary)
print(ruletaker_results.head(10))


{'examples': 300, 'parser_supported': 300, 'parser_coverage': np.float64(1.0), 'accuracy_all': np.float64(0.7166666666666667), 'accuracy_supported_only': np.float64(0.7166666666666667), 'total_inference_seconds': 0.26538540900000385, 'average_seconds_per_example': 0.0008846180300000128}
   id   gold  prediction  supported
0   0   True        True       True
1   1  False       False       True
2   2   True        True       True
3   3  False       False       True
4   4   True       False       True
5   5  False       False       True
6   6   True       False       True
7   7  False       False       True
8   8   True        True       True
9   9  False       False       True


In [11]:
# evaluating FOLIO

def folio_gold_to_binary(label):
    label = str(label).strip().lower()

    if label == "true":
        return True
    if label == "false":
        return False
    return None


def evaluate_folio(dataset):
    rows = []
    start = time.perf_counter()

    for i, example in enumerate(dataset):
        result = predict_entailment(
            str(example.get("premises", "")),
            str(example.get("conclusion", ""))
        )

        rows.append({
            "id": example.get("example_id", i),
            "gold": folio_gold_to_binary(example.get("label")),
            "prediction": result["prediction"],
            "supported": result["supported"]
        })

    elapsed = time.perf_counter() - start
    df = pd.DataFrame(rows)

    # Exclude Uncertain examples from binary accuracy.
    binary = df[df["gold"].notna()]
    supported = binary[binary["supported"]]

    summary = {
        "examples": len(df),
        "binary_examples": len(binary),
        "uncertain_examples": len(df) - len(binary),
        "parser_supported": int(binary["supported"].sum()),
        "parser_coverage": binary["supported"].mean() if len(binary) else 0,
        "accuracy_all": (binary["gold"] == binary["prediction"]).mean() if len(binary) else None,
        "accuracy_supported_only": (
            (supported["gold"] == supported["prediction"]).mean()
            if len(supported) else None
        ),
        "total_inference_seconds": elapsed
    }

    return summary, df


folio_summary, folio_results = evaluate_folio(folio_data)

print(folio_summary)
print(folio_results.head(10))


{'examples': 1001, 'binary_examples': 677, 'uncertain_examples': 324, 'parser_supported': 45, 'parser_coverage': np.float64(0.06646971935007386), 'accuracy_all': np.float64(0.31314623338257014), 'accuracy_supported_only': np.float64(0.4), 'total_inference_seconds': 0.45416737400000784}
     id   gold prediction  supported
0  1126   True      False      False
1  1127   True      False      False
2  1128  False      False      False
3  1129   True      False      False
4    20   None       None      False
5    21   True       None      False
6    22  False       None      False
7  1336   None      False      False
8  1337  False      False      False
9  1338   True      False      False


In [12]:
# inspecting counterbench

def counterbench_to_frame(data):
    frames = []

    for filename, payload in data.items():
        if isinstance(payload, list):
            frame = pd.DataFrame(payload)

        elif isinstance(payload, dict):
            lists = [v for v in payload.values() if isinstance(v, list)]
            frame = pd.DataFrame(lists[0]) if lists else pd.DataFrame([payload])

        else:
            continue

        if not frame.empty:
            frame["source_file"] = filename
            frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


counterbench_df = counterbench_to_frame(counterbench_data)

print("Columns:", counterbench_df.columns.tolist())
print("Rows:", len(counterbench_df))
display(counterbench_df.head())

if "type" in counterbench_df:
    display(counterbench_df["type"].value_counts(dropna=False))

if "answer" in counterbench_df:
    display(counterbench_df["answer"].value_counts(dropna=False))

Columns: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta', 'source_file']
Rows: 1200


,question,given_info,answer,type,question_id,meta,source_file
0,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf, Blaf causes Tr...",no,basic,0,"{'story_id': 'nonsense_0', 'rung': 3, 'query_t...",data_balanced_alpha_V1.json
1,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz, Razz causes Pe...",no,basic,200,"{'story_id': 'nonsense_1', 'rung': 3, 'query_t...",data_balanced_alpha_V1.json
2,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank, Vank causes Scu...",no,basic,400,"{'story_id': 'nonsense_2', 'rung': 3, 'query_t...",data_balanced_alpha_V1.json
3,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn, Blorn causes P...",no,basic,600,"{'story_id': 'nonsense_3', 'rung': 3, 'query_t...",data_balanced_alpha_V1.json
4,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes Splee, Splee causes Bl...",no,basic,800,"{'story_id': 'nonsense_4', 'rung': 3, 'query_t...",data_balanced_alpha_V1.json


,count
type,
basic,250
conditional,250
joint,250
nested,250
backdoor,200


,count
answer,
no,605
yes,595


In [13]:
# summary table of evaluation

summary_table = pd.DataFrame([
    {"dataset": "RuleTaker", **ruletaker_summary},
    {"dataset": "FOLIO", **folio_summary},
    {
        "dataset": "CounterBench",
        "examples": len(counterbench_df),
        "parser_supported": 0,
        "parser_coverage": 0.0,
        "accuracy_all": None,
        "accuracy_supported_only": None,
        "total_inference_seconds": None
    }
])

display(summary_table)

,dataset,examples,parser_supported,parser_coverage,accuracy_all,accuracy_supported_only,total_inference_seconds,average_seconds_per_example,binary_examples,uncertain_examples
0,RuleTaker,300,300,1.00000,0.716667,0.716667,0.265385,0.000885,NaN,NaN
1,FOLIO,1001,45,0.06647,0.313146,0.400000,0.454167,NaN,677.0,324.0
2,CounterBench,1200,0,0.00000,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# how is the ai reaching conclusion?

demo_context = """
Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful.
"""

trace_example = predict_entailment(demo_context, "Bob is successful.")

print("Prediction:", trace_example["prediction"])
print("Reason:", trace_example["reason"])

print("\nDerived facts:")
for fact in sorted(trace_example["derived_facts"], key=str):
    print("-", fact)

print("\nInference trace:")
for step in trace_example["trace"]:
    print(f"Round {step['round']}: {step['rule']}")
    print(f"  Substitution: {step['substitution']}")
    print(f"  Derived: {step['derived']}")

Prediction: False
Reason: Query checked against derived facts.

Derived facts:
- Kind(bob)
- Smart(bob)
- Smart_And_Kind_Then_They_Are_Successful(if_someone)
- Young(bob)

Inference trace:
Round 1: Young(?x) -> Smart(?x)
  Substitution: {'?x': 'bob'}
  Derived: Smart(bob)


In [15]:
def create_submission(
    evaluation_frame,
    id_column,
    context_column,
    query_column,
    output_path="submission.csv",
    unsupported_fallback=False
):
    required = [id_column, context_column, query_column]

    for col in required:
        if col not in evaluation_frame.columns:
            raise ValueError(f"Missing column: {col}")

    output_rows = []
    diagnostics = []

    for _, row in evaluation_frame.iterrows():
        result = predict_entailment(
            str(row[context_column]),
            str(row[query_column])
        )

        prediction = result["prediction"]
        if prediction is None:
            prediction = unsupported_fallback

        output_rows.append({
            "id": row[id_column],
            "predicted_label": bool(prediction)
        })

        diagnostics.append({
            "id": row[id_column],
            "supported": result["supported"],
            "reason": result["reason"]
        })

    submission = pd.DataFrame(output_rows)
    submission.to_csv(output_path, index=False)

    pd.DataFrame(diagnostics).to_csv(
        output_path.replace(".csv", "_diagnostics.csv"),
        index=False
    )

    print(f"Saved {output_path} with {len(submission)} predictions.")
    print("Diagnostics file also saved.")

    return submission

In [16]:
# this is for the private dataset


# evaluation_df = pd.read_csv("evaluation_input.csv")

# submission_df = create_submission(
#     evaluation_df,
#     id_column="id",
#     context_column="context",
#     query_column="question"
# )

# display(submission_df.head())

In [17]:
# save the data-set results to files

ruletaker_results.to_csv("ruletaker_public_dev_results.csv", index=False)
folio_results.to_csv("folio_public_dev_results.csv", index=False)
summary_table.to_csv("public_evaluation_summary.csv", index=False)

print("Saved public development results:")
print("- ruletaker_public_dev_results.csv")
print("- folio_public_dev_results.csv")
print("- public_evaluation_summary.csv")

Saved public development results:
- ruletaker_public_dev_results.csv
- folio_public_dev_results.csv
- public_evaluation_summary.csv


In [18]:
public_demo_submission = (
    ruletaker_results[["id", "prediction"]]
    .rename(columns={"prediction": "predicted_label"})
    .copy()
)
public_demo_submission["predicted_label"] = public_demo_submission["predicted_label"].astype(bool)
public_demo_submission.to_csv("submission.csv", index=False)

print("Created submission.csv as a PUBLIC RuleTaker development demonstration.")
print("Rows:", len(public_demo_submission))
display(public_demo_submission.head())


Created submission.csv as a PUBLIC RuleTaker development demonstration.
Rows: 300


,id,predicted_label
0,0,True
1,1,False
2,2,True
3,3,False
4,4,False
